# Many-Hamiltonian Resource Estimation
Loads HDF5 Hamiltonians from `hamlibs/`, builds Trotter circuits, transpiles to Clifford+T, runs Azure QDK then Qualtran on each.

In [ ]:
import sys, os, io, re, random, zipfile, warnings, pathlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4g}'.format)

from IPython.display import display, clear_output
from pathlib import Path
from contextlib import contextmanager
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.compiler import transpile as qk_transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import SuzukiTrotter

# add repo root to path so estimator imports work
_s = pathlib.Path.cwd()
for _ in range(8):
    if (_s / 'resourceEstimationPipeline').is_dir():
        REPO_ROOT = str(_s); break
    _s = _s.parent
else:
    REPO_ROOT = str(pathlib.Path.cwd().parent)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from resourceEstimationPipeline.config import PipelineConfig, TranspileConfig, AzureConfig, QualtranConfig
from resourceEstimationPipeline.estimators.azure import estimate as azure_estimate, apply_azure_to_qualtran
from resourceEstimationPipeline.estimators.qualtran import estimate as qt_estimate

print('REPO_ROOT:', REPO_ROOT)

In [ ]:
HAMLIB_DIR = '/Users/braden/Documents/hamlibs'
N_HAMS     = 10
TROTTER_T  = 1.0
TROTTER_K  = 10
ONLY_FILES = None

HAM_KEYS = [
    # Heisenberg (heis.hdf5)
    'graph-1D-grid-nonpbc-qubitnodes_Lx-10_h-1',
    'graph-2D-grid-pbc-qubitnodes_Lx-20_Ly-20_h-1',
    'graph-2D-hex-nonpbc-qubitnodes_Lx-10_Ly-10_h-1',
    'graph-2D-triag-pbc-qubitnodes_Lx-30_Ly-30_h-1',
    'graph-3D-grid-pbc-qubitnodes_Lx-10_Ly-10_Lz-10_h-1',
    # Fermi-Hubbard 2D (FH_D-2.hdf5)
    'fh-graph-2D-grid-nonpbc-qubitnodes_Lx-2_Ly-2_U-0_enc-jw',
    'fh-graph-2D-grid-nonpbc-qubitnodes_Lx-10_Ly-10_U-4_enc-jw',
    'fh-graph-2D-grid-nonpbc-qubitnodes_Lx-16_Ly-16_U-8_enc-jw',
    'fh-graph-2D-grid-nonpbc-qubitnodes_Lx-20_Ly-20_U-12_enc-jw',
    'fh-graph-2D-grid-nonpbc-qubitnodes_Lx-29_Ly-29_U-0_enc-jw',
]

cfg = PipelineConfig(
    transpile=TranspileConfig(optimization_level=1, seed_transpiler=42,
                              rotation_synthesis_enabled=False),
    azure=AzureConfig(error_budget=0.01, error_rate=1e-3, gate_time_ns=50.0,
                      measurement_time_ns=100.0, factory_type='RoundBased',
                      slow_down_factors=[1.0], optimization_level=1,
                      minimize='qubit_hours', pareto_index=0),
    qualtran=QualtranConfig(phys_err=1e-3, error_budget=0.01, data_block='compact',
                            factory_type='15to1', use_beverland=True,
                            use_azure_parameters=True, pareto_index=0),
)
print('HAMLIB_DIR:', HAMLIB_DIR)
print('Files found:', [f.name for f in Path(HAMLIB_DIR).glob('*.hdf5')] +
                      [f.name for f in Path(HAMLIB_DIR).glob('*.h5')])

In [ ]:
# self-contained helpers: scan HDF5 dir, parse Pauli strings, build circuits
_CT = ['h','t','tdg','s','sdg','x','y','z','cx','cz','ccx','swap','rz']

@contextmanager
def _open_hdf5(source):
    if isinstance(source, tuple):
        zp, inner = source
        with zipfile.ZipFile(zp) as zf:
            buf = io.BytesIO(zf.read(inner))
        with h5py.File(buf) as f:
            yield f
    else:
        with h5py.File(source) as f:
            yield f

def _scan_dir(directory, only_files=None):
    d, out = Path(directory), []
    for ext in ('*.hdf5', '*.h5'):
        for fp in sorted(d.glob(ext)):
            if only_files and fp.name not in only_files: continue
            with h5py.File(fp) as f:
                out.extend((fp, k) for k in f.keys())
    for zp in sorted(d.glob('*.zip')):
        if only_files and zp.name not in only_files: continue
        with zipfile.ZipFile(zp) as zf:
            for inner in zf.namelist():
                if inner.endswith(('.hdf5', '.h5')):
                    with h5py.File(io.BytesIO(zf.read(inner))) as f:
                        out.extend(((zp, inner), k) for k in f.keys())
    return out

def _parse_pauli(raw):
    idx = [int(m.group(1)) for m in re.finditer(r'[XYZI](\d+)', raw)]
    n = max(idx) + 1 if idx else 1
    terms = []
    for t in re.split(r' \+\n| \+ ', raw):
        t = t.strip()
        if not t: continue
        c, ops = t.split(' [', 1)
        coeff = complex(c.strip().strip('()')).real
        p = ['I'] * n
        for op in ops.rstrip(']').split():
            p[int(op[1:])] = op[0]
        terms.append((''.join(reversed(p)), coeff))
    return SparsePauliOp.from_list(terms), n

def load_circuits(directory, n_hams=10, trotter_t=1.0, trotter_k=10,
                  ham_keys=None, only_files=None):
    all_entries = _scan_dir(directory, only_files)
    if not all_entries:
        raise RuntimeError(f'No HDF5 files found in {directory}')
    if ham_keys:
        wanted = {k.lstrip('/') for k in ham_keys}
        entries = [(p, k) for p, k in all_entries if k in wanted]
    else:
        entries = random.sample(all_entries, min(n_hams, len(all_entries)))

    print(f'Loading {len(entries)} Hamiltonians from {directory}\n')
    out = []
    for i, (src, key) in enumerate(entries):
        tag = f'[{i+1}/{len(entries)}]'
        try:
            with _open_hdf5(src) as f:
                raw = f[key][()].decode()
            H, nq = _parse_pauli(raw)
            qc = QuantumCircuit(nq)
            qc.append(PauliEvolutionGate(H, time=trotter_t,
                       synthesis=SuzukiTrotter(order=1, reps=trotter_k)), range(nq))
            circ = qk_transpile(qc.decompose(), basis_gates=_CT, optimization_level=1)
            ops  = circ.count_ops()
            label = src[0].name if isinstance(src, tuple) else src.name
            out.append({
                'hdf5_file': label, 'ham_key': key, 'H': H, 'circuit': circ,
                'counts': {'n_qubits': nq, 'depth': circ.depth(),
                           'rz': ops.get('rz', 0),
                           't':  ops.get('t',  0) + ops.get('tdg', 0)},
            })
            print(f'  {tag} OK    {key}  ({nq}q  rz={ops.get("rz",0)}  depth={circ.depth()})')
        except Exception as e:
            print(f'  {tag} SKIP  {key}: {e}')
    print(f'\n{len(out)} circuits ready.')
    return out

In [ ]:
# ── 1. Load Hamiltonians ────────────────────────────────────────────────────
circuits = load_circuits(
    HAMLIB_DIR, n_hams=N_HAMS, trotter_t=TROTTER_T, trotter_k=TROTTER_K,
    ham_keys=HAM_KEYS, only_files=ONLY_FILES,
)
display(pd.DataFrame([
    {'ham_key': c['ham_key'], 'file': c['hdf5_file'], **c['counts']}
    for c in circuits
]))

In [ ]:
# ── 2. Run estimators — table updates live after each Hamiltonian ───────────
rows = []
for i, entry in enumerate(circuits):
    key, circuit, counts = entry['ham_key'], entry['circuit'], entry['counts']
    row = {'ham_key': key, 'file': entry['hdf5_file'],
           'n_qubits': counts['n_qubits'], 'rz': counts['rz'], 'depth': counts['depth']}

    az = None
    try:
        az = azure_estimate(circuit, cfg)
        row.update(az_qubits=az.physical_qubits, az_runtime_s=az.runtime_seconds,
                   az_t=az.t_count, az_d=az.code_distance, az_err=az.logical_error_rate)
    except Exception as e:
        row.update(az_qubits=None, az_runtime_s=None, az_t=None, az_d=None, az_err=None)
        print(f'Azure FAILED [{key}]: {e}')

    qt_cfg = apply_azure_to_qualtran(az, cfg) \
             if (az and cfg.qualtran.use_azure_parameters) else cfg
    try:
        qt = qt_estimate(circuit, qt_cfg)
        row.update(qt_qubits=qt.physical_qubits, qt_runtime_s=qt.runtime_seconds,
                   qt_t=qt.t_count, qt_d=qt.code_distance, qt_err=qt.logical_error_rate)
    except Exception as e:
        row.update(qt_qubits=None, qt_runtime_s=None, qt_t=None, qt_d=None, qt_err=None)
        print(f'Qualtran FAILED [{key}]: {e}')

    rows.append(row)
    df = pd.DataFrame(rows)
    df['qubit_ratio'] = df['qt_qubits'] / df['az_qubits']
    clear_output(wait=True)
    print(f'Progress: {i+1}/{len(circuits)}')
    display(df[['ham_key','n_qubits','rz',
                'az_qubits','qt_qubits','qubit_ratio',
                'az_d','qt_d','az_t','qt_t']])

print('Done.')

In [ ]:
# ── 3. Plots ────────────────────────────────────────────────────────────────
v = df.dropna(subset=['az_qubits','qt_qubits']).reset_index(drop=True)
labels = [k[:30] for k in v['ham_key']]
x, w = np.arange(len(v)), 0.35

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(v['az_qubits'], v['qt_qubits'], c='steelblue', edgecolors='white', s=80, zorder=3)
lim = max(v['az_qubits'].max(), v['qt_qubits'].max()) * 1.1
ax.plot([0,lim],[0,lim],'k--',lw=0.8,label='equal')
ax.set_xlabel('Azure qubits'); ax.set_ylabel('Qualtran qubits')
ax.set_title('Physical qubits'); ax.legend()

ax = axes[1]
ax.bar(x - w/2, v['az_qubits'],  w, label='Azure')
ax.bar(x + w/2, v['qt_qubits'], w, label='Qualtran')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Physical qubits'); ax.set_title('Qubits per Hamiltonian'); ax.legend()

ax = axes[2]
ax.hist(v['qubit_ratio'].dropna(), bins=12, color='steelblue', edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', lw=0.8, label='ratio=1')
ax.set_xlabel('Qualtran / Azure qubits'); ax.set_ylabel('Count')
ax.set_title('Qubit ratio distribution'); ax.legend()

plt.tight_layout()
plt.show()
print(f'Median qubit ratio: {v["qubit_ratio"].median():.3f}x')